## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted ✅')

## Step 2: Create Project Folder Structure

I will create all folders at once. Using `exist_ok=True` so that if I re-run this notebook later, it won't throw an error if the folders already exist.

In [ ]:
import os

# Root project folder in my Google Drive
PROJECT_ROOT = '/content/drive/MyDrive/Ocean_Plastic_Project'

# All sub-folders I need
folders = [
    'data/raw',
    'data/processed',
    'data/metadata',
    'notebooks',
    'visualizations',
    'scripts',
    'outputs'
]

# Create each folder
for folder in folders:
    full_path = os.path.join(PROJECT_ROOT, folder)
    os.makedirs(full_path, exist_ok=True)
    print(f'Created: {full_path} ✅')

print()
print('All project folders created successfully ✅')

In [ ]:
# Verify the structure was created
print('Project folder structure:')
for root, dirs, files in os.walk(PROJECT_ROOT):
    # Clean up the path for display
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')

## Step 3: Install Exact Library Versions

These are the 9 libraries required by the project plan, with exact versions pinned.

⚠️ **Note:** Google Colab already has most of these pre-installed. The `pip install` with `==version` will either confirm the right version is there or upgrade/downgrade to the exact one we need.

In [ ]:
# Install all 9 required libraries with exact version pins
# Running this as a single pip call is faster than 9 separate ones

!pip install -q \
    pandas==2.0.3 \
    geopandas==0.13.2 \
    scikit-learn==1.3.0 \
    numpy==1.24.3 \
    matplotlib==3.7.2 \
    seaborn==0.12.2 \
    folium==0.14.0 \
    openpyxl==3.1.2 \
    requests==2.31.0

print('Installation complete ✅')

## Step 4: Verify All Libraries — Check Exact Versions

This is the most important step. I don't just import — I print the version number and compare it to what was required.

In [ ]:
# Import all libraries and check versions
import pandas as pd
import geopandas as gpd
import sklearn
import numpy as np
import matplotlib
import seaborn as sns
import folium
import openpyxl
import requests

# Required versions (from project plan)
required = {
    'pandas':       '2.0.3',
    'geopandas':    '0.13.2',
    'scikit-learn': '1.3.0',
    'numpy':        '1.24.3',
    'matplotlib':   '3.7.2',
    'seaborn':      '0.12.2',
    'folium':       '0.14.0',
    'openpyxl':     '3.1.2',
    'requests':     '2.31.0',
}

# Installed versions
installed = {
    'pandas':       pd.__version__,
    'geopandas':    gpd.__version__,
    'scikit-learn': sklearn.__version__,
    'numpy':        np.__version__,
    'matplotlib':   matplotlib.__version__,
    'seaborn':      sns.__version__,
    'folium':       folium.__version__,
    'openpyxl':     openpyxl.__version__,
    'requests':     requests.__version__,
}

# Print comparison table
print(f'{"Library":<15} {"Required":<12} {"Installed":<12} {"Status"}')
print('-' * 55)
all_ok = True
for lib in required:
    req_ver = required[lib]
    inst_ver = installed[lib]
    status = '✅ OK' if inst_ver == req_ver else f'⚠️  MISMATCH'
    if inst_ver != req_ver:
        all_ok = False
    print(f'{lib:<15} {req_ver:<12} {inst_ver:<12} {status}')

print()
if all_ok:
    print('All 9 libraries verified at correct versions ✅')
else:
    print('⚠️  Some versions do not match — check above and re-install if needed')
    print('Mismatches may occur due to Colab pre-installed versions')
    print('Minor version differences (e.g. 2.0.3 vs 2.1.0) are usually fine')

## Step 5: Save Version Log to Metadata Folder

I'll save this version info to a text file so it's documented for my mentors and for WPR 1.

In [ ]:
import datetime

metadata_path = os.path.join(PROJECT_ROOT, 'data/metadata/environment_baseline.txt')

with open(metadata_path, 'w') as f:
    f.write('HCLTech Internship — Environment Baseline\n')
    f.write(f'Logged on: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write('=' * 50 + '\n\n')
    f.write('LIBRARY VERSIONS:\n')
    for lib, ver in installed.items():
        req = required[lib]
        match = 'OK' if ver == req else 'MISMATCH'
        f.write(f'  {lib:<15}: {ver:<12} (required: {req}) [{match}]\n')
    f.write('\nBOUNDING BOX:\n')
    f.write('  Latitude  : 5N to 30N\n')
    f.write('  Longitude : 65E to 95E\n')
    f.write('  Covers    : Arabian Sea, Bay of Bengal, Andaman Sea\n')
    f.write('\nPROJECT FOLDER STRUCTURE:\n')
    for folder in folders:
        f.write(f'  /Ocean_Plastic_Project/{folder}/\n')

print(f'Environment baseline saved to: {metadata_path} ✅')

# Print it to screen too
with open(metadata_path, 'r') as f:
    print(f.read())

## Step 6: Quick Functional Test

Don't just verify versions — make sure the key libraries actually **work** by running one operation from each.

In [ ]:
print('Running functional tests...')
print()

# Pandas — create a small dataframe
test_df = pd.DataFrame({'lat': [19.0, 13.0], 'lon': [72.8, 80.3], 'city': ['Mumbai', 'Chennai']})
assert len(test_df) == 2
print('pandas   — DataFrame creation          ✅')

# NumPy — array operations
arr = np.array([1, 2, 3, 4, 5])
assert arr.mean() == 3.0
print('numpy    — Array mean calculation       ✅')

# Matplotlib — just test the import (don't show a plot)
fig, ax = plt.subplots(1, 1, figsize=(2, 2))
ax.plot([1, 2], [3, 4])
plt.close()
print('matplotlib — Plot creation (closed)    ✅')

# Seaborn — load a test dataset
tips = sns.load_dataset('tips')
assert len(tips) > 0
print('seaborn  — Built-in dataset load        ✅')

# GeoPandas — create a GeoDataFrame
from shapely.geometry import Point
gdf = gpd.GeoDataFrame(
    test_df,
    geometry=[Point(lon, lat) for lat, lon in zip(test_df['lat'], test_df['lon'])],
    crs='EPSG:4326'
)
assert len(gdf) == 2
print('geopandas — GeoDataFrame creation      ✅')

# Scikit-learn — a basic scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled = scaler.fit_transform(np.array([[1.0], [2.0], [3.0]]))
assert scaled.shape == (3, 1)
print('scikit-learn — StandardScaler test     ✅')

# Folium — create a map object
m = folium.Map(location=[20, 80], zoom_start=5)
print('folium   — Map object creation          ✅')

# Openpyxl — create a workbook
from openpyxl import Workbook
wb = Workbook()
ws = wb.active
ws['A1'] = 'test'
assert ws['A1'].value == 'test'
print('openpyxl — Workbook cell write          ✅')

# Requests — just check it can make the object
s = requests.Session()
print('requests — Session creation             ✅')

print()
print('All 9 functional tests passed ✅')
print('Environment is fully operational and ready for Week 1 data work.')